# 09 — Blind-Test Evaluation

This notebook applies **already-trained** fold models from `05_model` to completely new
price + financials data that was never seen during training, feature-engineering, or
hyperparameter search.

### How it works

1. Accepts new `stock_prices_new.csv` and `financials_new.csv` (same schema as originals).
2. Runs the **identical** feature-engineering pipeline (steps from `02_eda`, `03_preprocessing`,
   `04_features`) but using **training-derived statistics** for clipping and nothing from the
   new data bleeds back into those constants.
3. Loads saved model(s) from disk and scores the new cross-sections.
4. Reports IC, ICIR, long-short return, and a full equity curve — all on data the model
   has never touched.

### Data leakage & overfitting audit notes

Findings from the pipeline review are documented in **Section 0** below.
This notebook is designed to be leakage-free by construction.

## 0. Pipeline Audit — Leakage & Overfitting Findings

A full review of notebooks 01–08 identified the following issues. **Items marked ✅ are already
fixed in the current notebooks. Items marked ⚠️ are still present and affect result validity.**

---

### ✅ Fixed — Look-ahead on disclosure date
The original pipeline merged financials on the same date they were disclosed, meaning the model
could see a filing on the same day the price moved. `03_preprocessing` now shifts the statement
`Date` forward by +1 calendar day before the `merge_asof`, so a filing on day T only enters
a price snapshot from day T+1 onward.

---

### ✅ Fixed — Target clipping using full-sample quantiles
`04_features` now computes clip bounds (1st / 99th percentile) using only rows before
`TRAIN_END = "2020-01-01"`, then applies the same bounds to all rows. Future return
distributions no longer influence how training targets are scaled.

---

### ✅ Fixed — Global future-leaking feature standardisation
Z-scoring is now done cross-sectionally date-by-date (groupby Date), so each month's
z-scores are computed using only same-month peers. No future dates contaminate the
scaling statistics.

---

### ✅ Fixed — Single fixed split calibrated to 2020 anomaly
Replaced by walk-forward expanding-window CV with a 1-month gap between training end
and test start.

---

### ⚠️ REMAINING — Profitability ratio clipping in `03_preprocessing` uses full-sample quantiles
In `03_preprocessing`, all profitability ratios (ROE, ROA, margins, growth features,
TTM ratios, forecast features) are clipped using quantiles computed over the **entire**
`statements` DataFrame before any train/test split:
```python
lower = statements[col].quantile(0.01)  # ← uses ALL dates including future test rows
upper = statements[col].quantile(0.99)
statements[col] = statements[col].clip(lower, upper)
```
This is a **mild but real** form of look-ahead: the clip bounds know the tails of the
future return distribution of fundamentals. The contamination is subtle (clipping, not
direct label leakage) but it means your reported IC is slightly optimistic. Fix: compute
clip bounds on training rows only, store them, and apply to test rows (same pattern as
done for `target_1m` in `04_features`).

---

### ⚠️ REMAINING — TTM rolling window crosses fiscal-year boundaries without time-ordering by DisclosedDate
In `02_eda`, the TTM (trailing twelve months) is computed as:
```python
statements.groupby('SecuritiesCode')[col].transform(lambda x: x.rolling(4, min_periods=4).sum())
```
This sorts by `DisclosedDate` but the rolling window uses integer position, not a time
index. If any filings are out of order or duplicated (which happens on the JPX dataset),
the TTM will silently include the wrong quarters. Use `.rolling('365D')` on a proper
DatetimeIndex, or at minimum assert no gaps > 1 quarter before the roll.

---

### ⚠️ REMAINING — Hyperparameter tuning objective sees all walk-forward folds
`08_hyperparameter_tuning` runs Optuna over the **same** walk-forward folds used for
final evaluation in `05_model`. The tuned hyperparameters are therefore adapted to the
specific realisation of those test folds. This is a form of indirect leakage: the model
architecture was chosen to maximise IC on what you call the OOS period. Proper practice
is a three-way split: an inner CV for tuning, a middle hold-out for model selection, and
a completely held-out blind test set. This notebook implements that last tier.

---

### ⚠️ REMAINING — Raw balance sheet levels (Equity, TotalAssets) may leak cross-sectionally
As noted in earlier diagnostics, raw level features (absolute yen values of Equity,
TotalAssets, etc.) are not comparable across companies and are partially addressed by
size-quintile neutralisation. However they are still present in `feature_cols` and the
model can learn to use absolute yen magnitudes as a size proxy that survives
neutralisation. Recommend dropping raw levels in favour of their ratio forms only.

---

### ⚠️ REMAINING — Early stopping inner window may be too small for pairwise ranker
The inner validation window is the last 6 months of the training slice. For the first
fold (train through 2019-12), this is ~6 months × ~2000 stocks = ~12 000 rows, but
XGBRanker groups are per-date (one group per month, so 6 groups). NDCG with 6 groups
is noisy enough that early stopping fires at iteration 0 or very early, effectively
returning an untrained model. Evidence: `best_iteration=0` in fold logs. Fix: either
disable early stopping and use a fixed `n_estimators` chosen via hyperparameter search,
or extend the inner validation window to ≥12 months.

---

### Summary table

| Issue | Status | Severity |
|-------|--------|----------|
| Same-day disclosure look-ahead | ✅ Fixed | High |
| Target clipping with future data | ✅ Fixed | High |
| Global future-leaking feature z-score | ✅ Fixed | High |
| Single split on 2020 anomaly | ✅ Fixed | High |
| Feature clipping with full-sample quantiles | ⚠️ Present | Medium |
| TTM rolling on unordered/gapped quarters | ⚠️ Present | Medium |
| HP tuning objective = eval objective | ⚠️ Present | Medium |
| Raw balance-sheet level features | ⚠️ Present | Low-Medium |
| Early stopping on 6-group inner window | ⚠️ Present | High |

In [115]:
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from xgboost import XGBRanker

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.4f}".format

## 1. Configuration

Point these paths at your new (blind) data and at the artefacts saved by `05_model`.

In [116]:
# ── New blind data ─────────────────────────────────────────────────────────────
# Must have identical column schemas to the original CSVs.
NEW_PRICES_PATH      = r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\blind\stock_prices_new.csv"
NEW_FINANCIALS_PATH  = r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\blind\financials_new.csv"

# ── Artefacts from the training run ───────────────────────────────────────────
# feature_cols.json  — list of feature column names (from 03_preprocessing output)
FEATURE_COLS_PATH    = r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processed\feature_cols.json"

# Clipping bounds saved from the training pipeline (see Section 3 below for format).
# If you don't have this file yet, run Section 3a to generate it from your original data.
CLIP_BOUNDS_PATH     = r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processed\clip_bounds.json"

# Saved XGBRanker models — one .pkl per walk-forward fold, e.g. model_fold_0.pkl
# If you saved them as JSON use XGBRanker().load_model(path) instead.
MODEL_DIR            = r"C:\Users\naksh\Projects\xgboost_stock_ranking\models"

# ── Target column ─────────────────────────────────────────────────────────────
# 'Target' in the raw price file is the next-month return used as ground truth.
# If your blind data doesn't have this column, set HAS_TARGET = False.
HAS_TARGET           = True
TARGET_COL           = "target_1m"    # name after the merge step

# ── Size quintile proxy (must exist after feature construction) ────────────────
SIZE_PROXY           = "avg_dollar_vol_1m"
N_QUINTILES          = 5

## 2. Load blind data

In [117]:
prices     = pd.read_csv(NEW_PRICES_PATH,     parse_dates=["Date"])
financials = pd.read_csv(NEW_FINANCIALS_PATH, parse_dates=["Date", "DisclosedDate"], low_memory=False)

print("Blind prices shape    :", prices.shape)
print("Blind financials shape:", financials.shape)
print("Price date range:", prices["Date"].min().date(), "->", prices["Date"].max().date())

Blind prices shape    : (24040, 12)
Blind financials shape: (1215, 45)
Price date range: 2017-01-04 -> 2021-12-03


In [118]:
financials["SecuritiesCode"] = pd.to_numeric(financials["SecuritiesCode"], errors="coerce")
financials = financials.dropna(subset=["SecuritiesCode"])
financials["SecuritiesCode"] = financials["SecuritiesCode"].astype("int64")

## 3a. (One-time) Generate clip_bounds.json from original training data

Run this cell **once** against your original `statements_quarterized.csv` to serialise
the training-set clip bounds. Then comment it out — the blind test always loads from the
saved file, never recomputes from new data.

This cell fixes the **⚠️ Feature clipping with full-sample quantiles** issue noted in
Section 0: bounds are computed on training rows only and stored for reproducible
application to any future data.

In [119]:
# ── UNCOMMENT AND RUN ONCE, THEN RECOMMENT ────────────────────────────────────
# 
# TRAIN_END = "2020-01-01"   # must match 04_features TRAIN_END

# statements_orig = pd.read_csv(
#     r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\statements_quarterized.csv"
# )
# statements_orig["DisclosedDate"] = pd.to_datetime(statements_orig["DisclosedDate"])
# train_stmts = statements_orig[statements_orig["DisclosedDate"] < TRAIN_END]

# CLIP_COLS = [
#     "roe", "roa", "profit_margin", "operating_margin", "asset_turnover",
#     "equity_ratio", "operating_roa", "ordinary_roa", "operating_roe",
#     "ordinary_roe", "ordinary_margin", "profit_to_assets", "sales_to_equity",
#     "ttm_roa", "ttm_roe", "ttm_operating_roa", "ttm_operating_roe",
#     "ttm_profit_margin", "ttm_operating_margin", "ttm_ordinary_margin",
#     "ttm_asset_turnover", "ttm_sales_to_equity", "ttm_profit_to_assets",
#     "forecast_sales_growth", "forecast_profit_growth", "forecast_eps_growth",
#     "forecast_operating_margin", "forecast_profit_margin", "forecast_roa", "forecast_roe",
#     "q_NetSales_yoy", "q_Profit_yoy", "TotalAssets_yoy", "Equity_yoy",
# ]

# clip_bounds = {}
# for col in CLIP_COLS:
#     if col in train_stmts.columns:
#         clip_bounds[col] = {
#             "lower": float(train_stmts[col].quantile(0.01)),
#             "upper": float(train_stmts[col].quantile(0.99)),
#         }

# with open(CLIP_BOUNDS_PATH, "w") as f:
#     json.dump(clip_bounds, f, indent=2)

# print(f"Saved {len(clip_bounds)} clip bounds to {CLIP_BOUNDS_PATH}")

print("Section 3a skipped — loading from saved clip_bounds.json")

Section 3a skipped — loading from saved clip_bounds.json


## 3b. Load training artefacts

In [120]:
with open(FEATURE_COLS_PATH) as f:
    feature_cols = json.load(f)

with open(CLIP_BOUNDS_PATH) as f:
    clip_bounds = json.load(f)

print(f"Loaded {len(feature_cols)} feature columns")
print(f"Loaded clip bounds for {len(clip_bounds)} features")

Loaded 79 feature columns
Loaded clip bounds for 0 features


## 4. Load saved models

Models must have been saved after the `05_model` training loop. Example save code (add to 05):
```python
import pickle, os
os.makedirs(MODEL_DIR, exist_ok=True)
for i, m in enumerate(trained_models):
    with open(f"{MODEL_DIR}/model_fold_{i}.pkl", "wb") as f:
        pickle.dump(m, f)
```

In [121]:
import glob, os

model_paths = sorted(glob.glob(os.path.join(MODEL_DIR, "model_fold_*.pkl")))

if not model_paths:
    raise FileNotFoundError(
        f"No model_fold_*.pkl files found in {MODEL_DIR}.\n"
        "Add pickle.dump(...) for each fold in 05_model and rerun it."
    )

models = []
for path in model_paths:
    with open(path, "rb") as f:
        models.append(pickle.load(f))

print(f"Loaded {len(models)} fold model(s):")
for p in model_paths:
    print(" ", os.path.basename(p))

Loaded 4 fold model(s):
  model_fold_0.pkl
  model_fold_1.pkl
  model_fold_2.pkl
  model_fold_3.pkl


## 5. Feature engineering on blind data

Mirrors `02_eda` → `03_preprocessing` → `04_features` exactly, but uses
training-derived clip bounds (from `clip_bounds.json`) instead of recomputing
from the blind data.

In [122]:
# ── 5.1 Filter statement types (mirrors 02_eda) ───────────────────────────────

KEEP_DOCS = [
    "ForecastRevision",
    "1QFinancialStatements_Consolidated_JP",
    "2QFinancialStatements_Consolidated_JP",
    "3QFinancialStatements_Consolidated_JP",
    "FYFinancialStatements_Consolidated_JP",
]

NUMERIC_COLS = [
    "NetSales", "OperatingProfit", "OrdinaryProfit", "Profit",
    "TotalAssets", "Equity",
    "BookValuePerShare", "EarningsPerShare",
    "ForecastNetSales", "ForecastOperatingProfit", "ForecastOrdinaryProfit",
    "ForecastProfit", "ForecastEarningsPerShare",
    "AverageNumberOfShares",
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock",
    "NumberOfTreasuryStockAtTheEndOfFiscalYear",
    "EquityToAssetRatio",
]

price_codes = set(prices["SecuritiesCode"].unique())

statements = (
    financials[
        financials["TypeOfDocument"].isin(KEEP_DOCS) &
        financials["SecuritiesCode"].isin(price_codes)
    ]
    .sort_values(["SecuritiesCode", "Date"])
    .copy()
)

statements[NUMERIC_COLS] = statements[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")

STATEMENT_DOCS = [
    "1QFinancialStatements_Consolidated_JP",
    "2QFinancialStatements_Consolidated_JP",
    "3QFinancialStatements_Consolidated_JP",
    "FYFinancialStatements_Consolidated_JP",
]

statements_q = statements[statements["TypeOfDocument"].isin(STATEMENT_DOCS)].copy()

print("Filtered statements:", statements_q.shape)

Filtered statements: (308, 45)


In [123]:
# ── 5.2 Quarterize cumulative income-statement values (mirrors 02_eda) ─────────

QUARTER_MAP = {
    "1QFinancialStatements_Consolidated_JP": 1,
    "2QFinancialStatements_Consolidated_JP": 2,
    "3QFinancialStatements_Consolidated_JP": 3,
    "FYFinancialStatements_Consolidated_JP": 4,
}

statements_q["quarter"]     = statements_q["TypeOfDocument"].map(QUARTER_MAP)
statements_q["fiscal_year"] = pd.to_datetime(statements_q["CurrentPeriodEndDate"]).dt.year
statements_q = statements_q.sort_values(["SecuritiesCode", "fiscal_year", "quarter"])

FLOW_COLS = ["NetSales", "OperatingProfit", "OrdinaryProfit", "Profit"]

prev_quarter = (
    statements_q
    .groupby(["SecuritiesCode", "fiscal_year"])["quarter"]
    .shift(1)
)

for col in FLOW_COLS:
    statements_q[f"q_{col}"] = np.nan
    q1_mask    = statements_q["quarter"] == 1
    valid_mask = (statements_q["quarter"] > 1) & (prev_quarter == statements_q["quarter"] - 1)
    statements_q.loc[q1_mask,    f"q_{col}"] = statements_q.loc[q1_mask, col]
    statements_q.loc[valid_mask, f"q_{col}"] = (
        statements_q.loc[valid_mask, col]
        - statements_q.groupby(["SecuritiesCode", "fiscal_year"])[col].shift(1).loc[valid_mask]
    )

# TTM
ttm_base = ["q_NetSales", "q_OperatingProfit", "q_OrdinaryProfit", "q_Profit"]
statements_q = statements_q.sort_values(["SecuritiesCode", "DisclosedDate"])
for col in ttm_base:
    statements_q[f"{col}_ttm"] = (
        statements_q
        .groupby("SecuritiesCode")[col]
        .transform(lambda x: x.rolling(4, min_periods=4).sum())
    )

print("Quarterized shape:", statements_q.shape)

Quarterized shape: (308, 55)


In [124]:
# ── 5.3 Profitability, growth, and forecast ratios (mirrors 03_preprocessing) ─

def safe_ratio(num, den):
    return np.where(den == 0, np.nan, num / den)

s = statements_q  # alias

s["roe"]                = safe_ratio(s["q_Profit"],              s["Equity"])
s["roa"]                = safe_ratio(s["q_Profit"],              s["TotalAssets"])
s["profit_margin"]      = safe_ratio(s["q_Profit"],              s["q_NetSales"])
s["operating_margin"]   = safe_ratio(s["q_OperatingProfit"],     s["q_NetSales"])
s["asset_turnover"]     = safe_ratio(s["q_NetSales"],            s["TotalAssets"])
s["equity_ratio"]       = safe_ratio(s["Equity"],                s["TotalAssets"])
s["operating_roa"]      = safe_ratio(s["q_OperatingProfit"],     s["TotalAssets"])
s["ordinary_roa"]       = safe_ratio(s["q_OrdinaryProfit"],      s["TotalAssets"])
s["operating_roe"]      = safe_ratio(s["q_OperatingProfit"],     s["Equity"])
s["ordinary_roe"]       = safe_ratio(s["q_OrdinaryProfit"],      s["Equity"])
s["ordinary_margin"]    = safe_ratio(s["q_OrdinaryProfit"],      s["q_NetSales"])
s["profit_to_assets"]   = safe_ratio(s["q_Profit"],              s["TotalAssets"])
s["sales_to_equity"]    = safe_ratio(s["q_NetSales"],            s["Equity"])

s["ttm_roa"]               = safe_ratio(s["q_Profit_ttm"],           s["TotalAssets"])
s["ttm_roe"]               = safe_ratio(s["q_Profit_ttm"],           s["Equity"])
s["ttm_operating_roa"]     = safe_ratio(s["q_OperatingProfit_ttm"],  s["TotalAssets"])
s["ttm_operating_roe"]     = safe_ratio(s["q_OperatingProfit_ttm"],  s["Equity"])
s["ttm_profit_margin"]     = safe_ratio(s["q_Profit_ttm"],           s["q_NetSales_ttm"])
s["ttm_operating_margin"]  = safe_ratio(s["q_OperatingProfit_ttm"],  s["q_NetSales_ttm"])
s["ttm_ordinary_margin"]   = safe_ratio(s["q_OrdinaryProfit_ttm"],   s["q_NetSales_ttm"])
s["ttm_asset_turnover"]    = safe_ratio(s["q_NetSales_ttm"],         s["TotalAssets"])
s["ttm_sales_to_equity"]   = safe_ratio(s["q_NetSales_ttm"],         s["Equity"])
s["ttm_profit_to_assets"]  = safe_ratio(s["q_Profit_ttm"],           s["TotalAssets"])

for growth_col in ["q_NetSales", "q_Profit", "TotalAssets", "Equity"]:
    s[f"{growth_col}_yoy"] = (
        s.groupby("SecuritiesCode")[growth_col].pct_change(4)
    )

s["forecast_operating_margin"] = safe_ratio(s["ForecastOperatingProfit"], s["ForecastNetSales"])
s["forecast_profit_margin"]    = safe_ratio(s["ForecastProfit"],          s["ForecastNetSales"])
s["forecast_roa"]              = safe_ratio(s["ForecastProfit"],          s["TotalAssets"])
s["forecast_roe"]              = safe_ratio(s["ForecastProfit"],          s["Equity"])
s["forecast_sales_growth"]     = safe_ratio(s["ForecastNetSales"],        s["NetSales"]) - 1
s["forecast_profit_growth"]    = safe_ratio(s["ForecastProfit"],          s["Profit"]) - 1
s["forecast_eps_growth"]       = safe_ratio(s["ForecastEarningsPerShare"], s["EarningsPerShare"]) - 1

s = s.replace([np.inf, -np.inf], np.nan)

# Apply training-derived clip bounds (fixes the ⚠️ issue from Section 0)
for col, bounds in clip_bounds.items():
    if col in s.columns:
        s[col] = s[col].clip(bounds["lower"], bounds["upper"])

statements_q = s
print("Ratios computed and clipped using training bounds.")

Ratios computed and clipped using training bounds.


In [125]:
# ── 5.4 Merge prices with statements (mirrors 03_preprocessing merge_asof) ────
# Key: disclosure embargo — DisclosedDate + 1 day before merge.

prices = prices.sort_values(["SecuritiesCode", "Date"])

statements_q["merge_date"] = statements_q["DisclosedDate"] + pd.Timedelta(days=1)
statements_q = statements_q.sort_values(["SecuritiesCode", "merge_date"])

# Resample prices to monthly (last trading day of each month)
prices_monthly = (
    prices
    .set_index("Date")
    .groupby("SecuritiesCode")
    .resample("ME")  # month-end
    .last()
    .drop(columns=["SecuritiesCode"], errors="ignore")
    .reset_index()
)

# Build momentum, volatility, and volume features from daily prices
prices_feat = prices.copy()
prices_feat = prices_feat.sort_values(["SecuritiesCode", "Date"])

# Monthly return windows (1m, 3m, 6m, 12m) — use Close as adjusted proxy
for w in [1, 3, 6, 12]:
    prices_feat[f"mom_{w}m"] = (
        prices_feat.groupby("SecuritiesCode")["Close"]
        .transform(lambda x: x.pct_change(w * 21))  # ~21 trading days/month
    )

# Volatility (21d and 63d rolling std of daily returns)
prices_feat["daily_ret"] = prices_feat.groupby("SecuritiesCode")["Close"].pct_change()
for w, label in [(21, "1m"), (63, "3m")]:
    prices_feat[f"vol_{label}"] = (
        prices_feat.groupby("SecuritiesCode")["daily_ret"]
        .transform(lambda x: x.rolling(w, min_periods=max(5, w // 2)).std())
    )

# Dollar volume
prices_feat["dollar_vol"] = prices_feat["Close"] * prices_feat["Volume"]
for w, label in [(21, "1m"), (63, "3m")]:
    prices_feat[f"avg_dollar_vol_{label}"] = (
        prices_feat.groupby("SecuritiesCode")["dollar_vol"]
        .transform(lambda x: x.rolling(w, min_periods=5).mean())
    )

# Snap to month-end
price_mom_cols = ["SecuritiesCode", "Date", "Close", "AdjustmentFactor", "SupervisionFlag",
                  "mom_1m", "mom_3m", "mom_6m", "mom_12m",
                  "vol_1m", "vol_3m",
                  "avg_dollar_vol_1m", "avg_dollar_vol_3m"]
price_mom_cols = [c for c in price_mom_cols if c in prices_feat.columns]

monthly_prices = (
    prices_feat[price_mom_cols]
    .groupby("SecuritiesCode")
    .resample("ME", on="Date")
    .last()
    .reset_index()
)

if HAS_TARGET:
    monthly_prices[TARGET_COL] = (
        monthly_prices.groupby("SecuritiesCode")["Close"]
        .transform(lambda x: x.pct_change().shift(-1))
    )

print("Monthly price features shape:", monthly_prices.shape)

Monthly price features shape: (1200, 14)


In [128]:
# ── 5.5 merge_asof statements onto monthly price snapshots ─────────────────────

merged_parts = []
for code, price_grp in monthly_prices.groupby("SecuritiesCode"):
    stmt_grp = statements_q[statements_q["SecuritiesCode"] == code].sort_values("merge_date")
    if stmt_grp.empty:
        continue
    merged = pd.merge_asof(
        price_grp.sort_values("Date"),
        stmt_grp,
        left_on="Date",
        right_on="merge_date",
        by="SecuritiesCode",
        direction="backward",
    )
    merged_parts.append(merged)

monthly = pd.concat(merged_parts, ignore_index=True)
monthly = monthly.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

print("Merged monthly shape:", monthly.shape)

KeyError: 'Date'

In [ ]:
# ── 5.6 Feature preparation (mirrors 04_features) ─────────────────────────────
# Impute, then z-score with size-quintile neutralisation.
# NO statistics are recomputed from blind data — all scaling is cross-sectional
# (per date), which is inherently leak-free.

if HAS_TARGET:
    monthly = monthly.dropna(subset=[TARGET_COL])

# Keep only feature columns that actually exist after the merge
available_features = [f for f in feature_cols if f in monthly.columns]
missing_features   = [f for f in feature_cols if f not in monthly.columns]

if missing_features:
    print(f"WARNING: {len(missing_features)} feature(s) missing from blind data; "
          f"will be zero-filled:\n  {missing_features[:10]}{'...' if len(missing_features)>10 else ''}")
    for col in missing_features:
        monthly[col] = 0.0

# Cross-sectional median imputation (leak-free: uses only same-date peers)
monthly[feature_cols] = (
    monthly
    .groupby("Date")[feature_cols]
    .transform(lambda x: x.fillna(x.median()))
)
monthly[feature_cols] = monthly[feature_cols].fillna(0)

# Size quintile assignment
monthly["size_quintile"] = (
    monthly
    .groupby("Date")[SIZE_PROXY]
    .transform(lambda x: pd.qcut(x.rank(method="first"), q=N_QUINTILES, labels=False))
)

# Feature buckets (must match 04_features exactly)
PRICE_EXACT = {
    "log_market_cap", "price_to_earnings", "earnings_yield",
    "dividend_yield", "forecast_dividend_yield", "treasury_share_ratio",
}
PRICE_PREFIXES = [
    "mom_", "ma_", "vol_", "skew_", "max_return_",
    "avg_dollar_vol_", "rel_volume", "AdjustmentFactor", "SupervisionFlag",
]
PRICE_FEATURES = [
    f for f in feature_cols
    if (f in PRICE_EXACT or any(f.startswith(p) for p in PRICE_PREFIXES))
]
FUNDAMENTAL_FEATURES = [f for f in feature_cols if f not in PRICE_FEATURES]

monthly[FUNDAMENTAL_FEATURES] = (
    monthly
    .groupby(["Date", "size_quintile"])[FUNDAMENTAL_FEATURES]
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
)
monthly[PRICE_FEATURES] = (
    monthly
    .groupby("Date")[PRICE_FEATURES]
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
)

monthly = monthly.drop(columns=["size_quintile"])

print("Final blind dataset shape:", monthly.shape)
print("Date range:", monthly["Date"].min().date(), "->", monthly["Date"].max().date())

## 6. Score the blind data

We ensemble the fold models by averaging their raw scores. This is the most conservative
approach — each fold model was trained on a different expanding window, so the ensemble
represents the "average model" over the CV period.

In [ ]:
X_blind = monthly[feature_cols]

score_matrix = np.column_stack([
    model.predict(X_blind) for model in models
])

monthly["score"] = score_matrix.mean(axis=1)

print("Score distribution:")
print(monthly["score"].describe())

## 7. Evaluation

All metrics below are computed on data the model never touched during training,
hyperparameter search, or feature normalisation.

In [ ]:
if not HAS_TARGET:
    print("HAS_TARGET=False — skipping IC / return metrics. Scores saved to blind_scores.csv.")
    monthly[["Date", "SecuritiesCode", "score"]].to_csv(
        r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\blind\blind_scores.csv", index=False
    )
else:
    # ── Information Coefficient ────────────────────────────────────────────────
    monthly_ic = (
        monthly.groupby("Date")
               .apply(lambda x: spearmanr(x["score"], x[TARGET_COL]).correlation,
                      include_groups=False)
    )

    icir        = monthly_ic.mean() / (monthly_ic.std() + 1e-8)
    annual_icir = icir * np.sqrt(12)

    print("=" * 50)
    print("BLIND TEST — IC Summary")
    print("=" * 50)
    print(monthly_ic.describe().to_string())
    print(f"\nMean IC       : {monthly_ic.mean():.4f}")
    print(f"ICIR          : {icir:.4f}")
    print(f"Annualised ICIR: {annual_icir:.4f}")
    print(f"IC > 0        : {(monthly_ic > 0).mean():.2%}")

In [ ]:
if HAS_TARGET:
    # ── Long-short return ──────────────────────────────────────────────────────
    def long_short_return(df, pct=0.10):
        df = df.sort_values("score")
        n  = max(1, int(len(df) * pct))
        return df.iloc[-n:][TARGET_COL].mean() - df.iloc[:n][TARGET_COL].mean()

    portfolio = monthly.groupby("Date").apply(long_short_return, include_groups=False)

    print("\n" + "=" * 50)
    print("BLIND TEST — Long-Short Portfolio (top/bottom 10%)")
    print("=" * 50)
    print(portfolio.describe().to_string())
    print(f"\nAvg monthly L/S return  : {portfolio.mean():.4%}")
    print(f"Annualised Sharpe (approx): {portfolio.mean() / portfolio.std() * np.sqrt(12):.2f}")

In [ ]:
if HAS_TARGET:
    # ── IC by size quintile ────────────────────────────────────────────────────
    tmp = monthly.copy()
    tmp["size_q"] = (
        tmp.groupby("Date")[SIZE_PROXY]
           .transform(lambda x: pd.qcut(x.rank(method="first"), 5, labels=False, duplicates="drop"))
    )

    ic_size = (
        tmp.groupby(["Date", "size_q"])
           .apply(lambda x: spearmanr(x["score"], x[TARGET_COL]).correlation,
                  include_groups=False)
           .groupby(level=1)
           .mean()
    )

    # ── Plots ──────────────────────────────────────────────────────────────────
    equity = (1 + portfolio).cumprod()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    equity.plot(ax=axes[0], title="Blind Test — L/S Equity Curve", color="steelblue")
    axes[0].set_ylabel("Cumulative return")
    axes[0].axhline(1, color="black", linewidth=0.8, linestyle="--")
    axes[0].grid(True)

    monthly_ic.plot(ax=axes[1], alpha=0.5, label="Monthly IC", color="steelblue")
    monthly_ic.rolling(3).mean().plot(ax=axes[1], label="3m rolling mean",
                                       linewidth=2, color="darkblue")
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title("Blind Test — Information Coefficient")
    axes[1].legend()
    axes[1].grid(True)

    ic_size.plot.bar(ax=axes[2], color="steelblue", edgecolor="white")
    axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[2].set_title("Blind Test — IC by Size Quintile")
    axes[2].set_ylabel("Mean IC")
    axes[2].set_xlabel("Size quintile (0=small, 4=large)")
    axes[2].grid(True, axis="y")

    plt.tight_layout()
    plt.savefig(
        r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\blind\blind_test_results.png",
        dpi=150, bbox_inches="tight"
    )
    plt.show()
    print("\nIC by size quintile:")
    print(ic_size)

## 8. Save results

In [ ]:
out_dir = r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\blind"

save_cols = ["Date", "SecuritiesCode", "score"]
if HAS_TARGET:
    save_cols.append(TARGET_COL)

monthly[save_cols].to_csv(f"{out_dir}\\blind_scores.csv", index=False)
print("Saved blind_scores.csv")

if HAS_TARGET:
    summary = {
        "mean_ic":        float(monthly_ic.mean()),
        "icir":           float(icir),
        "annual_icir":    float(annual_icir),
        "ic_pct_positive": float((monthly_ic > 0).mean()),
        "ls_return_mean": float(portfolio.mean()),
        "ls_sharpe_annual": float(portfolio.mean() / portfolio.std() * np.sqrt(12)),
        "n_months":       int(len(monthly_ic)),
    }
    with open(f"{out_dir}\\blind_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print("Saved blind_summary.json")
    print(json.dumps(summary, indent=2))